# 02 - Day 1: OWLv2 ingredient detection test
1. Runtime > Change runtime type > **T4 GPU**.
2. Put 10-15 JPG/PNG photos in Google Drive: `MyDrive/AaharAI/photos/`.
3. Run all cells. Then fill in `labels.csv` (created next to the photos) and re-run the last cell.
Do not save this notebook back to GitHub.

In [ ]:
GITHUB_USER = "OnYourLeftover"

from google.colab import drive
drive.mount("/content/drive")
!pip -q install rapidfuzz
import sys
!git -C /content/AaharAI pull || git clone https://github.com/{GITHUB_USER}/AaharAI.git /content/AaharAI
sys.path.insert(0, "/content/AaharAI")
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from pathlib import Path
from PIL import Image
from torchvision.ops import nms
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from src import vocab

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
assert dev == "cuda", "Switch the runtime to a T4 GPU (Runtime > Change runtime type)."

PHOTOS = Path("/content/drive/MyDrive/AaharAI/photos")
imgs = sorted(p for p in PHOTOS.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(len(imgs), "photos found in", PHOTOS)

name = "google/owlv2-base-patch16-ensemble"
proc = Owlv2Processor.from_pretrained(name)
model = Owlv2ForObjectDetection.from_pretrained(name).to(dev).eval()

PROMPT_FIX = {"brinjal - all varieties": "brinjal (eggplant)", "celery stalk": "celery", "spring onions": "spring onion",
              "button mushroom": "mushroom", "peas": "green peas", "french beans": "green beans",
              "yogurt": "bowl of curd (yogurt)", "capsicum": "capsicum (bell pepper)", "coriander leaves": "coriander leaves (cilantro)"}
NAMES = sorted(vocab.DETECT)
PROMPTS = [f"a photo of {PROMPT_FIX.get(n, n)}" for n in NAMES]
print(len(NAMES), "ingredient prompts")

In [ ]:
# one forward pass per photo; thresholds are applied afterwards so the sweep is cheap
cache = {}
for p in imgs:
    img = Image.open(p).convert("RGB"); img.thumbnail((1600, 1600))
    inputs = proc(text=[PROMPTS], images=img, return_tensors="pt").to(dev)
    with torch.no_grad():
        cache[p.name] = (model(**inputs), max(img.size))
print("done:", len(cache), "photos")

def decode(photo, thr=0.2, iou=0.5):
    out, s = cache[photo]
    res = proc.post_process_object_detection(out, threshold=thr, target_sizes=torch.tensor([[s, s]]).to(dev))[0]
    if len(res["scores"]) == 0:
        return {}
    keep = nms(res["boxes"], res["scores"], iou_threshold=iou)      # class-agnostic NMS
    best = {}
    for i in keep.tolist():
        n = NAMES[int(res["labels"][i])]
        best[n] = max(best.get(n, 0.0), float(res["scores"][i]))
    return dict(sorted(best.items(), key=lambda kv: -kv[1]))

for photo in cache:
    print(photo, "->", {k: round(v, 2) for k, v in decode(photo, 0.2).items()})

In [ ]:
import pandas as pd
LABELS = PHOTOS / "labels.csv"      # columns: image, ingredients (semicolon-separated names from vocab.DETECT)
if not LABELS.exists():
    pd.DataFrame({"image": list(cache), "ingredients": ""}).to_csv(LABELS, index=False)
    print("Created", LABELS, "- fill the 'ingredients' column (e.g. onion;tomato;potato), then re-run this cell.")
else:
    lab = pd.read_csv(LABELS).fillna("")
    truth = {r.image: {x.strip() for x in str(r.ingredients).split(";") if x.strip()} for r in lab.itertuples()}
    truth = {k: v for k, v in truth.items() if v}
    print(len(truth), "labelled photos")
    for thr in (0.1, 0.15, 0.2, 0.3):
        tp = fp = fn = 0
        for photo, t in truth.items():
            if photo not in cache: continue
            pred = set(decode(photo, thr))
            tp += len(pred & t); fp += len(pred - t); fn += len(t - pred)
        P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1); F = 2 * P * R / max(P + R, 1e-9)
        print(f"threshold {thr:<5} precision {P:.2f}  recall {R:.2f}  F1 {F:.2f}   (tp {tp}, fp {fp}, fn {fn})")
    # which ingredients does it miss / invent most?
    from collections import Counter
    miss, fake = Counter(), Counter()
    for photo, t in truth.items():
        if photo not in cache: continue
        pred = set(decode(photo, 0.2))
        miss.update(t - pred); fake.update(pred - t)
    print("\nmost missed at 0.2:", miss.most_common(8))
    print("most invented at 0.2:", fake.most_common(8))